In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
from neuro import config
from os.path import join
sys.path.append(join(config.REPO_DIR, 'experiments'))

import dvu
import seaborn as sns
import os
import pandas as pd
from copy import deepcopy
from matplotlib import pyplot as plt
import numpy as np
from neuro import config
import imodelsx.process_results
import neuro.features.qa_questions as qa_questions
import joblib
from tqdm import tqdm
import neuro.viz
from neuro import analyze_helper, viz
fit_encoding = __import__('02_fit_encoding')
dvu.set_style()

results_dir = '/home/chansingh/mntv1/deep-fMRI/encoding/oct5_2025_linear_pc_ablation'
rr, cols_varied, mets = analyze_helper.load_clean_results(results_dir)

/home/chansingh/automated-brain-explanations/.venv/lib/python3.12/site-packages/spacy/cli/_util.py:23: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
/home/chansingh/automated-brain-explanations/.venv/lib/python3.12/site-packages/weasel/util/config.py:8: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
100%|██████████| 192/192 [03:40<00:00,  1.15s/it]

experiment varied these params: ['subject', 'pc_components', 'feature_space', 'embedding_layer', 'qa_embedding_model', 'qa_questions_version']



/home/chansingh/imodelsX/imodelsx/process_results.py:99: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[k] = df[k].fillna(np.nan)


In [30]:
rr = rr[rr.subject.isin(['S01', 'S02', 'S03'])]
rr = rr[~rr.feature_space.str.contains('llama')]

In [31]:
rtab = rr.pivot_table(index=['feature_space', 'embedding_layer'], columns=['subject', 'pc_components'], values='corrs_test_mean')

# add average subject col when pc components = -1
rtab[('AVG', -1)] = rtab.xs(-1, level='pc_components', axis=1).mean(axis=1)

# add avg when pc components == 100
rtab[('AVG', 100)] = rtab.xs(100, level='pc_components', axis=1).mean(axis=1)

In [32]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(
        rtab
        .style.background_gradient(cmap='viridis', axis=0)
        .format(precision=3)
    )

In [33]:
print(rtab.style.format(precision=3).to_latex(hrules=True))

\begin{tabular}{llrrrrrrrr}
\toprule
 & subject & \multicolumn{2}{r}{S01} & \multicolumn{2}{r}{S02} & \multicolumn{2}{r}{S03} & \multicolumn{2}{r}{AVG} \\
 & pc_components & -1 & 100 & -1 & 100 & -1 & 100 & -1 & 100 \\
feature_space & embedding_layer &  &  &  &  &  &  &  &  \\
\midrule
eng1000 & -1 & 0.049 & 0.054 & 0.090 & 0.092 & 0.099 & 0.108 & 0.079 & 0.085 \\
qa_embedder & -1 & 0.053 & 0.055 & 0.106 & 0.112 & 0.116 & 0.122 & 0.091 & 0.096 \\
\bottomrule
\end{tabular}



In [34]:
# corrs test mean was 0.046716, 0.105916, 0.144678 for llama best
np.mean([0.046716, 0.105916, 0.144678])

np.float64(0.09910333333333332)